In [ ]:
"""
============================================================
10_campaign_analysis.py
------------------------------------------------------------
Purpose:
Evaluate uplift after campaign execution.

Inputs:
    data/campaign_results.csv

Outputs:
    outputs/campaign_summary.csv
    outputs/decile_analysis.csv
    outputs/uplift_by_decile.png
    outputs/overall_uplift.png

============================================================
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import chi2_contingency

In [ ]:
# ==========================================================
# Paths
# ==========================================================

BASE_DIR = Path.cwd().parent

In [ ]:
# ---------------------------------------------------------
# Load campaign results
# ---------------------------------------------------------

df = pd.read_csv(f"{BASE_DIR}/data/campaign_results.csv")

print("="*60)
print("Campaign Results Loaded")
print("="*60)
print(df.head())

In [ ]:
# ---------------------------------------------------------
# Basic Summary
# ---------------------------------------------------------

summary = df.groupby("Group").agg(
    Clients=("Client_Code","count"),
    Converted=("converted","sum"),
    Conversion_Rate=("converted","mean"),
    Avg_Propensity=("propensity_score","mean")
)

print(summary)

summary.to_csv(f"{BASE_DIR}/outputs/campaign_summary.csv")

In [ ]:
# ---------------------------------------------------------
# Uplift Calculation
# ---------------------------------------------------------

treatment_rate = summary.loc["Treatment","Conversion_Rate"]
control_rate = summary.loc["Control","Conversion_Rate"]

absolute_uplift = treatment_rate - control_rate

relative_uplift = (
    absolute_uplift / control_rate
) * 100

print("\n")
print("="*60)
print("OVERALL UPLIFT")
print("="*60)

print(f"Control Conversion     : {control_rate:.4%}")
print(f"Treatment Conversion   : {treatment_rate:.4%}")

print(f"Absolute Uplift        : {absolute_uplift:.4%}")
print(f"Relative Uplift        : {relative_uplift:.2f}%")

In [ ]:
# ---------------------------------------------------------
# Statistical Significance
# ---------------------------------------------------------

contingency = pd.crosstab(
    df["Group"],
    df["converted"]
)

chi2,p,_,_ = chi2_contingency(contingency)

print("\nChi-square Test")
print("----------------------------")
print("P-value :",p)

if p < 0.05:
    print("Result : Statistically Significant")
else:
    print("Result : Not Significant")

In [ ]:
# ---------------------------------------------------------
# Decile Level Analysis
# ---------------------------------------------------------

decile = (
    df
    .groupby(["Propensity_Decile","Group"])
    .agg(
        Clients=("Client_Code","count"),
        Converted=("converted","sum"),
        Conversion_Rate=("converted","mean")
    )
    .reset_index()
)

pivot = decile.pivot(
    index="Propensity_Decile",
    columns="Group",
    values="Conversion_Rate"
)

pivot["Absolute_Uplift"] = (
    pivot["Treatment"] -
    pivot["Control"]
)

pivot["Relative_Uplift_%"] = (
    pivot["Absolute_Uplift"] /
    pivot["Control"]
) * 100

pivot.to_csv(f"{BASE_DIR}/outputs/decile_analysis.csv")

print("\n")
print("="*60)
print("UPLIFT BY DECILE")
print("="*60)
print(pivot)

In [ ]:
# ---------------------------------------------------------
# Plot 1
# ---------------------------------------------------------

plt.figure(figsize=(10,5))

plt.plot(
    pivot.index,
    pivot["Treatment"],
    marker="o",
    label="Treatment"
)

plt.plot(
    pivot.index,
    pivot["Control"],
    marker="o",
    label="Control"
)

plt.xlabel("Propensity Decile")
plt.ylabel("Conversion Rate")

plt.title("Conversion Rate by Decile")

plt.legend()

plt.grid(True)

plt.tight_layout()

plt.savefig(f"{BASE_DIR}/outputs/uplift_by_decile.png")

plt.close()

In [ ]:
# ---------------------------------------------------------
# Plot 2
# ---------------------------------------------------------

plt.figure(figsize=(6,5))

plt.bar(
    ["Control","Treatment"],
    [control_rate,treatment_rate]
)

plt.ylabel("Conversion Rate")

plt.title("Overall Campaign Performance")

plt.tight_layout()

plt.savefig(f"{BASE_DIR}/outputs/overall_uplift.png")

plt.close()

print("\nCharts Saved.")

In [ ]:
# ---------------------------------------------------------
# Business Metrics
# ---------------------------------------------------------

print("\n")
print("="*60)
print("BUSINESS METRICS")
print("="*60)

print(f"Total Clients               : {len(df):,}")

print(f"Treatment Clients           : {(df.Group=='Treatment').sum():,}")

print(f"Control Clients             : {(df.Group=='Control').sum():,}")

print(f"Treatment Conversion Rate   : {treatment_rate:.2%}")

print(f"Control Conversion Rate     : {control_rate:.2%}")

print(f"Absolute Uplift             : {absolute_uplift:.2%}")

print(f"Relative Uplift             : {relative_uplift:.2f}%")

print(f"P-value                     : {p:.6f}")

print("\nAnalysis Complete.")